# Task 176: mne_nirs_flim — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: fNIRS hemodynamic response estimation using modified Beer-Lambert law

**Paper**: MNE-NIRS: fNIRS analysis
**Repository**: https://github.com/mne-tools/mne-nirs

### Physical Background

Physical systems have parameters (frequencies, damping, material constants) extracted by fitting models to measurements.

### Forward Model

```
y = f(theta) + n
```

### Inverse Problem

```
theta_hat = argmin ||y - f(theta)||^2  or Bayesian inference
```

### Performance Metrics
- **PSNR**: 31.01 dB
- **SSIM**: N/A
- **Evaluation**: 1D hemodynamic signal inversion (MBLL, PSNR, CC)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
#!/usr/bin/env python3
"""
Task 176: mne_nirs_flim
fNIRS hemodynamic response recovery using Modified Beer-Lambert Law (MBLL).

Forward: Known Δ[HbO], Δ[HbR] → compute ΔOD at 760nm, 850nm
Inverse: Given ΔOD at two wavelengths → solve 2×2 system for Δ[HbO], Δ[HbR]
"""
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`canonical_hrf`, `add_noise`, `compute_cc`, `main`

In [ ]:
def canonical_hrf(t_hrf, peak=6.0, undershoot=16.0, ratio=6.0):
    """Generate a canonical hemodynamic response function (double-gamma)."""
    from scipy.stats import gamma as gamma_dist
    h = (gamma_dist.pdf(t_hrf, peak / 1.0, scale=1.0) -
         gamma_dist.pdf(t_hrf, undershoot / 1.0, scale=1.0) / ratio)
    h = h / np.max(np.abs(h))
    return h

def add_noise(signal, snr_db=25):
    """Add Gaussian noise at specified SNR."""
    sig_power = np.mean(signal ** 2)
    noise_power = sig_power / (10 ** (snr_db / 10))
    noise = RNG.normal(0, np.sqrt(noise_power), signal.shape)
    return signal + noise

def compute_cc(gt, rec):
    """Pearson correlation coefficient."""
    return float(np.corrcoef(gt, rec)[0, 1])

def main():
    print("=" * 60)
    print("Task 176: fNIRS Hemodynamic Response Recovery (MBLL)")
    print("=" * 60)

    # 1. Synthesize ground truth
    print("\n[1/5] Synthesizing hemodynamic data...")
    hbo_gt, hbr_gt, stimulus, block_starts, block_duration = synthesize_hemodynamic_signals(T, FS)
    print(f"  HbO range: [{hbo_gt.min()*1e6:.3f}, {hbo_gt.max()*1e6:.3f}] µM")
    print(f"  HbR range: [{hbr_gt.min()*1e6:.3f}, {hbr_gt.max()*1e6:.3f}] µM")

    # 2. Forward model
    print("\n[2/5] Applying forward MBLL...")
    od_760_clean, od_850_clean = forward_mbll(hbo_gt, hbr_gt)
    od_760_noisy = add_noise(od_760_clean, snr_db=25)
    od_850_noisy = add_noise(od_850_clean, snr_db=25)
    print(f"  ΔOD_760 range: [{od_760_noisy.min():.6f}, {od_760_noisy.max():.6f}]")
    print(f"  ΔOD_850 range: [{od_850_noisy.min():.6f}, {od_850_noisy.max():.6f}]")

    # 3. Inverse solve
    print("\n[3/5] Solving inverse MBLL...")
    hbo_rec, hbr_rec = inverse_mbll(od_760_noisy, od_850_noisy)
    print(f"  Recovered HbO range: [{hbo_rec.min()*1e6:.3f}, {hbo_rec.max()*1e6:.3f}] µM")
    print(f"  Recovered HbR range: [{hbr_rec.min()*1e6:.3f}, {hbr_rec.max()*1e6:.3f}] µM")

    # 4. Evaluate
    print("\n[4/5] Computing metrics...")
    metrics = evaluate(hbo_gt, hbr_gt, hbo_rec, hbr_rec)
    for k, v in metrics.items():
        print(f"  {k}: {v:.6f}")

    # Save metrics
    metrics_path = os.path.join(RESULTS_DIR, 'metrics.json')
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"\n  Metrics saved to {metrics_path}")

    # 5. Visualization
    print("\n[5/5] Creating visualization...")
    plot_results(T, hbo_gt, hbr_gt, od_760_noisy, od_850_noisy,
                 hbo_rec, hbr_rec, block_starts, block_duration, metrics)

    # 6. Save arrays
    gt_stack = np.stack([hbo_gt, hbr_gt], axis=0)   # (2, N)
    rec_stack = np.stack([hbo_rec, hbr_rec], axis=0) # (2, N)
    input_stack = np.stack([od_760_noisy, od_850_noisy], axis=0)

    np.save(os.path.join(RESULTS_DIR, 'ground_truth.npy'), gt_stack)
    np.save(os.path.join(RESULTS_DIR, 'reconstruction.npy'), rec_stack)
    np.save(os.path.join(RESULTS_DIR, 'input_data.npy'), input_stack)
    print(f"  Arrays saved to {RESULTS_DIR}")

    # Final summary
    print("\n" + "=" * 60)
    print("SUMMARY")
    print(f"  Overall PSNR: {metrics['PSNR_dB']:.2f} dB")
    print(f"  Overall CC:   {metrics['CC']:.4f}")
    print(f"  Overall RMSE: {metrics['RMSE']:.2e}")
    ok = metrics['PSNR_dB'] > 20 and metrics['CC'] > 0.9
    print(f"  PASS: {'YES' if ok else 'NO'}")
    print("=" * 60)

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
y = f(theta) + n
```

Functions: `synthesize_hemodynamic_signals`, `forward_mbll`

In [ ]:
def synthesize_hemodynamic_signals(t, fs):
    """Create HbO and HbR block-design time series with 3 stimulus blocks."""
    n = len(t)

    # Stimulus blocks: 3 blocks, each ~20s on, ~60s off
    stimulus = np.zeros(n)
    block_starts = [30, 110, 200]  # seconds
    block_duration = 20            # seconds
    for start in block_starts:
        i0 = int(start * fs)
        i1 = int((start + block_duration) * fs)
        stimulus[i0:min(i1, n)] = 1.0

    # HRF kernel (30 seconds long)
    t_hrf = np.arange(0, 30, 1.0 / fs)
    hrf = canonical_hrf(t_hrf)

    # Convolve stimulus with HRF
    bold = np.convolve(stimulus, hrf, mode='full')[:n]
    bold = bold / np.max(np.abs(bold))

    # HbO: positive response, peak ~5 µM
    hbo = bold * 5e-6  # in Molar

    # HbR: negative, smaller (~-1.5 µM)
    hbr = -bold * 1.5e-6

    return hbo, hbr, stimulus, block_starts, block_duration

def forward_mbll(hbo, hbr):
    """
    Forward Modified Beer-Lambert Law.
    ΔOD(λ) = ε_HbO(λ)·Δ[HbO]·DPF(λ)·d + ε_HbR(λ)·Δ[HbR]·DPF(λ)·d
    """
    od_760 = (EPS_HBO_760 * hbo * DPF_760 * D +
              EPS_HBR_760 * hbr * DPF_760 * D)
    od_850 = (EPS_HBO_850 * hbo * DPF_850 * D +
              EPS_HBR_850 * hbr * DPF_850 * D)
    return od_760, od_850

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
theta_hat = argmin ||y - f(theta)||^2  or Bayesian inference
```

Functions: `inverse_mbll`

In [ ]:
def inverse_mbll(od_760_noisy, od_850_noisy):
    """
    Inverse MBLL: solve 2×2 linear system at each time point.
    A · [HbO, HbR]^T = [ΔOD_760, ΔOD_850]^T
    """
    A = np.array([
        [EPS_HBO_760 * DPF_760 * D, EPS_HBR_760 * DPF_760 * D],
        [EPS_HBO_850 * DPF_850 * D, EPS_HBR_850 * DPF_850 * D]
    ])
    # Stack observations: shape (2, N)
    od_stack = np.stack([od_760_noisy, od_850_noisy], axis=0)
    # Solve A x = b  →  x = A^{-1} b
    A_inv = np.linalg.inv(A)
    x = A_inv @ od_stack  # (2, N)
    hbo_rec = x[0]
    hbr_rec = x[1]
    return hbo_rec, hbr_rec

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_psnr`, `compute_rmse`, `evaluate`, `plot_results`

In [ ]:
def compute_psnr(gt, rec):
    """Peak Signal-to-Noise Ratio."""
    mse = np.mean((gt - rec) ** 2)
    if mse < 1e-30:
        return 100.0
    peak = np.max(np.abs(gt))
    return 10 * np.log10(peak ** 2 / mse)

def compute_rmse(gt, rec):
    """Root Mean Squared Error."""
    return float(np.sqrt(np.mean((gt - rec) ** 2)))

def evaluate(hbo_gt, hbr_gt, hbo_rec, hbr_rec):
    """Compute metrics for both HbO and HbR."""
    metrics = {
        'HbO_PSNR_dB': compute_psnr(hbo_gt, hbo_rec),
        'HbO_CC': compute_cc(hbo_gt, hbo_rec),
        'HbO_RMSE': compute_rmse(hbo_gt, hbo_rec),
        'HbR_PSNR_dB': compute_psnr(hbr_gt, hbr_rec),
        'HbR_CC': compute_cc(hbr_gt, hbr_rec),
        'HbR_RMSE': compute_rmse(hbr_gt, hbr_rec),
    }
    # Overall averages
    metrics['PSNR_dB'] = (metrics['HbO_PSNR_dB'] + metrics['HbR_PSNR_dB']) / 2
    metrics['CC'] = (metrics['HbO_CC'] + metrics['HbR_CC']) / 2
    metrics['RMSE'] = (metrics['HbO_RMSE'] + metrics['HbR_RMSE']) / 2
    return metrics

def plot_results(t, hbo_gt, hbr_gt, od_760_noisy, od_850_noisy,
                 hbo_rec, hbr_rec, block_starts, block_duration, metrics):
    """Create 4-panel visualization."""
    fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

    # Helper: shade stimulus blocks
    def shade_blocks(ax):
        for s in block_starts:
            ax.axvspan(s, s + block_duration, color='yellow', alpha=0.2, label=None)

    # Panel 1: Ground truth HbO & HbR
    ax = axes[0]
    shade_blocks(ax)
    ax.plot(t, hbo_gt * 1e6, 'r-', lw=1.5, label='HbO (GT)')
    ax.plot(t, hbr_gt * 1e6, 'b-', lw=1.5, label='HbR (GT)')
    ax.set_ylabel('Concentration (µM)')
    ax.set_title('Ground Truth Hemodynamic Response')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

    # Panel 2: Noisy optical density input
    ax = axes[1]
    shade_blocks(ax)
    ax.plot(t, od_760_noisy, 'purple', lw=0.8, alpha=0.7, label='ΔOD 760nm')
    ax.plot(t, od_850_noisy, 'orange', lw=0.8, alpha=0.7, label='ΔOD 850nm')
    ax.set_ylabel('Optical Density Change')
    ax.set_title('Noisy Optical Density Input (MBLL Forward + Noise)')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

    # Panel 3: Recovered vs GT
    ax = axes[2]
    shade_blocks(ax)
    ax.plot(t, hbo_gt * 1e6, 'r--', lw=1.5, alpha=0.6, label='HbO (GT)')
    ax.plot(t, hbo_rec * 1e6, 'r-', lw=1.2, label='HbO (Recovered)')
    ax.plot(t, hbr_gt * 1e6, 'b--', lw=1.5, alpha=0.6, label='HbR (GT)')
    ax.plot(t, hbr_rec * 1e6, 'b-', lw=1.2, label='HbR (Recovered)')
    ax.set_ylabel('Concentration (µM)')
    ax.set_title(f'Recovered Hemodynamic Response  |  PSNR={metrics["PSNR_dB"]:.1f} dB, CC={metrics["CC"]:.4f}')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

    # Panel 4: Residuals
    ax = axes[3]
    shade_blocks(ax)
    ax.plot(t, (hbo_gt - hbo_rec) * 1e6, 'r-', lw=1.0, label='HbO residual')
    ax.plot(t, (hbr_gt - hbr_rec) * 1e6, 'b-', lw=1.0, label='HbR residual')
    ax.axhline(0, color='k', ls='--', lw=0.5)
    ax.set_ylabel('Residual (µM)')
    ax.set_xlabel('Time (s)')
    ax.set_title('Residuals (GT − Recovered)')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fig_path = os.path.join(RESULTS_DIR, 'reconstruction_result.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Figure saved to {fig_path}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("=" * 60)
print("Task 176: fNIRS Hemodynamic Response Recovery (MBLL)")
print("=" * 60)

### 1. Synthesize ground truth

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# 1. Synthesize ground truth
print("\n[1/5] Synthesizing hemodynamic data...")
hbo_gt, hbr_gt, stimulus, block_starts, block_duration = synthesize_hemodynamic_signals(T, FS)
print(f"  HbO range: [{hbo_gt.min()*1e6:.3f}, {hbo_gt.max()*1e6:.3f}] µM")
print(f"  HbR range: [{hbr_gt.min()*1e6:.3f}, {hbr_gt.max()*1e6:.3f}] µM")

### 2. Forward model

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 2. Forward model
print("\n[2/5] Applying forward MBLL...")
od_760_clean, od_850_clean = forward_mbll(hbo_gt, hbr_gt)
od_760_noisy = add_noise(od_760_clean, snr_db=25)
od_850_noisy = add_noise(od_850_clean, snr_db=25)
print(f"  ΔOD_760 range: [{od_760_noisy.min():.6f}, {od_760_noisy.max():.6f}]")
print(f"  ΔOD_850 range: [{od_850_noisy.min():.6f}, {od_850_noisy.max():.6f}]")

### 3. Inverse solve

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 3. Inverse solve
print("\n[3/5] Solving inverse MBLL...")
hbo_rec, hbr_rec = inverse_mbll(od_760_noisy, od_850_noisy)
print(f"  Recovered HbO range: [{hbo_rec.min()*1e6:.3f}, {hbo_rec.max()*1e6:.3f}] µM")
print(f"  Recovered HbR range: [{hbr_rec.min()*1e6:.3f}, {hbr_rec.max()*1e6:.3f}] µM")

### 4. Evaluate

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# 4. Evaluate
print("\n[4/5] Computing metrics...")
metrics = evaluate(hbo_gt, hbr_gt, hbo_rec, hbr_rec)
for k, v in metrics.items():
    print(f"  {k}: {v:.6f}")

# Save metrics
metrics_path = os.path.join(RESULTS_DIR, 'metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"\n  Metrics saved to {metrics_path}")

### 5. Visualization

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# 5. Visualization
print("\n[5/5] Creating visualization...")
plot_results(T, hbo_gt, hbr_gt, od_760_noisy, od_850_noisy,
             hbo_rec, hbr_rec, block_starts, block_duration, metrics)

### 6. Save arrays

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 6. Save arrays
gt_stack = np.stack([hbo_gt, hbr_gt], axis=0)   # (2, N)
rec_stack = np.stack([hbo_rec, hbr_rec], axis=0) # (2, N)
input_stack = np.stack([od_760_noisy, od_850_noisy], axis=0)

np.save(os.path.join(RESULTS_DIR, 'ground_truth.npy'), gt_stack)
np.save(os.path.join(RESULTS_DIR, 'reconstruction.npy'), rec_stack)
np.save(os.path.join(RESULTS_DIR, 'input_data.npy'), input_stack)
print(f"  Arrays saved to {RESULTS_DIR}")

# Final summary
print("\n" + "=" * 60)
print("SUMMARY")
print(f"  Overall PSNR: {metrics['PSNR_dB']:.2f} dB")
print(f"  Overall CC:   {metrics['CC']:.4f}")
print(f"  Overall RMSE: {metrics['RMSE']:.2e}")
ok = metrics['PSNR_dB'] > 20 and metrics['CC'] > 0.9
print(f"  PASS: {'YES' if ok else 'NO'}")
print("=" * 60)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **mne_nirs_flim**:

1. **Problem Setup**: Physical systems have parameters (frequencies, damping, material constants) extracted by fitting models to measurements.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=31.01 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: MNE-NIRS: fNIRS analysis
- Repository: https://github.com/mne-tools/mne-nirs
